# HOME

## **Introdução** 

O modelo Meridian Angstrom de rating do especialista do Microcrédito foi desenvolvido com o objetivo de servir de insumo para o time de políticas do produto na tarefa de avaliação de risco de concessão de crédito ao cliente. Além disso, ele também possui como finalidade permitir a discriminação dos bons e maus especialistas não só do ponto de vista de gestão de sua carteira de clientes mas também a partir de informações demográficas, informações de outros CNPJs relacionados ao parceiro (através do vínculo de participação societária), informações de parentesco existente entre especialistas e subvendedores (quando se trata de PJ ampliada), entre outros. O público a ser escorado pelo modelo é representado por todos os especialistas credenciados junto a operação de Microcrédito em determinada referência de escoragem que se encontrem com status 'Habilitado a Produzir'. 

# DATA PREP

## Definição do público
--- 

### **Público Alvo** 

O modelo tem como público alvo da escoragem os especialistas do produto de Microcrédito, que consistem em PJs que atuam no papel de intermedidores entre o banco e o cliente final na venda do produto. Estes PJs possuem cadastro junto a operação de Microcrédito e
seu rating/classificação mensal interfere seja na variabilidade de valores e prazos das ofertas que ele tem disponível para venda no mês vigente, seja na sua remuneração direta. Um breve acompanhamento da evolução da volumetria total mensal de especialistas ativos cadastrados junto a operação pode ser observada no gráfico abaixo:

(GRÁFICO VOLUMETRIA ESPEC/SSAFRA)


### **Histórico de Desenvolvimento** 

Para desenvolvimento e treino do modelo foram utilizadas as safras de contratação de julho/2023 a janeiro/2024. Já para avaliação do desempenho do modelo, foram utilizadas as safras de fevereiro/2024 e março/2024. A quantidade reduzida de referências utilizadas no processo de desenvolvimento do modelo se deve, entre outros, ao pequeno volume de dados existentes, consequência do fato de a própria operação de Microcrédito não possuir um número muito grande de especialistas cadastrados (qnd comparado a patamares que costumamos associar ao tamanho de conjuntos de dados de treinamento/teste de modelos de machine learning). Além disso, outros 2 fatores que contribuíram para o numero reduzido de referências utilizadas foram a disponibilidade do histórico de dados (não possuíamos histórico prévio a 2022) e a definição do target do modelo que, como tratado na seção específica sobre o tema (link Target), necessita acompanhar uma métrica de performance por no mínimo 3 meses a partir da referência avaliada para conseguir definir o label da variável resposta

### **Fontes de Informação** 

O público é construído de forma simples, utilizando a tabela de Gestão do Parceiro (db_source_repositoriosdedados_microcreditoanalytics_spec_01.tbma4_gestao_parceiro) onde são disponibilizadas as informações mais recentes e completas dos especialistas cadastrados junto a operação. 

Uma peculiaridade do processo é o fato de que, em alguns casos, mais de uma pessoa acaba atuando como especialista, porém atrelado ao mesmo CNPJ de cadastro. A esta configuração, deu-se o nome de PJ Ampliada, pelo fato do identificador único corresponder a um grupo de pessoas. Portanto, aqui torna-se importante salientar que, mesmo com a existëncia da configuração de PJ Ampliada, o rating do modelo é atribuído a nível de CNPJ, fazendo com que todos os subvendedores de uma mesma PJ Ampliada contribuam para o valor final obtido e que a todos seja atribuído o mesmo rating, tratando-os como uma única entidade. A chave única do púbico do modelo acaba sendo, então, o CNPJ/IDEQ3 do Especialista (ou grupo de subvendedores) e a safra a qual as demais informações do registro se referem.

### **Público Final de Desenvolvimento** 

In [ ]:
# Definição do público ← ## **Público Alvo** ## **público de desenvolvimento** ### **Histórico de desenvolvimento** Para desenvolvimento e treino do modelo foram utilizadas as safras de contratação de julho/2023 a janeiro/2024. Já para avaliação do desempenho do modelo, foram utilizadas as safras de fevereiro/2024 e março/2024. A quantidade reduzida de referências utilizadas no process de desenvolvimento do modelo se deve, entre outros, ao pequeno volume de dados existentes, consequência do fato de a própria operação de Microcrédito não possuir um número muito grande de especialistas cadastrados (qnd comparado a patamares que costumamos associar ao tamanho de conjuntos de dados de treinamento/teste de modelos de machine learning). Além disso, outros 2 fatores que contribuíram para o numero reduzido de referências utilizadas foram a disponibilidade do histórico de dados (não possuíamos histórico prévio a 2022) e a definição do target do modelo que, como tratado na seção específica sobre o tema (link Target), necessita acompanhar uma métrica de performance por по mínimo 3 meses a partir da referência avaliada para conseguir definir o label da variável resposta ### **Fontes** O público é construído de forma simples, utilizando a tabela de Gestão do Parceiro (db_source_repositoriosdedados_microcreditoanalytics_spec_01.tbma4_gestao_parceiro) onde são disponibilizadas as informações mais recentes completas dos especialistas cadastrados junto a operação. Uma peculiaridade do processo é o fato de que, en alguns casos, mais de uma pessoa acaba atuando como especialista, porém atrelado ao mesmo CNPJ de cadastro. A esta configuracão. deu-se o nome de PJ Ampliada. pelo fato do identificador único corresponder a um erupo de vessoas. Cookie Preferences Ln 1, Col 1 Spaces: 4 1 audience.mo

## Variável Resposta
---

### **Definição do Target**

Para definição do conceito e marcação do **target** (ou variável resposta) utilizado no processo de modelagem do Angstrom foi feito uso da classificação pré existente do especialista. 

[Regras atribuição classificação especialista](../data/0_artifacts/target_dev_deals_orig.png) 

A tabela acima ilustra as regras utilizadas nesta classificação. Como é possível observar, os especialistas eram classificados em uma dentre as 7 categorias existentes e os critérios utilizados para tal se concentravam em refletir a inadimplência da sua carteira de clientes no curto prazo (máximo 3 meses anteriores, usando NPL e FPD). O rating do especialista, portanto, tratava-se de uma avaliação pós fato destes indicadores.

Com o objetivo de trazer um viés preditivo ao modelo desenvolvido, decidimos tentar abordar variáveis que expandissem nosso entendimento do especialista em si. Propusemos features para o modelo que procurassem refletir a qualidade da gestão que o especialista faz da própria carteira, bem como alguns dados sobre outras atividades profissionais exercidas por ele, seja na mesma área de atuação (Microcrédito) seja como sócio/proprietário de alguma PJ em qualquer outra área, e também alguns dados demográficos que pudessem caracterizar seu contexto socioeconômico.

Tomamos a decisão de modelar a probabilidade de o especialista vir a assumir o rating 'ddd' (pior rating da classificação anterior) em uma janela de até 3 meses. Temos, portanto:

... ==> target = 0
... ==> target = 1

Para marcação do target foi utilizada a base de classificacao dos especialistas, cruzada com a base de público


para utilização no presente modelo foi a performance 90 em 12 (atraso de 90 dias ou mais em um período de 12 meses). Para determinação da contratação interna de crédito imobiliário foi utilizado como indicador a presença de um número de contrato atrelado à proposta na base de propostas do produto disponível no SAS. O cruzamento da base de propostas com a base de contratação auxiliou na validação da informação. Para determinação da contratação de crédito imobiliário no mercado foram utilizadas as bases de [Contratação](https://confluence- itau.tecnologia.prod.ops.aws.cloud.ihf/x/kmP5Lw), (Performance Interna] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/x/WWP5Lw) e  do projeto [Foucault] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/pages/viewpage.action?pageId=799108541) que consolidam as informações contidas nas principais bases de público e performance (interna e mercado) que possuímos à disposição.

### **Bad Rate**

# HOME
---
## Introdução
## Papéis e responsáveis


# DATA PREP
---
## Base de Público
### - Público Alvo
### - Histórico de Desenvolvimento
### - Fontes de informação
### - Público Final de Desenvolvimento (amostragem)

## Variável Resposta
### - Definição do Target
### - Bad Rate

## Variáveis explicativas
### - Fontes de informação exploradas
### - Pré seleção de variáveis


### - Extração de variáveis de books
Algumas variáveis mapeadas na listagem acima encontravam-se já disponíveis nos books de variáveis da NPM e, portanto, prontas para consumo em ambiente produtivo. Para estas variáveis, bastou a extração dos valores referentes aos registros da base de público, que foi efetuada através de uma query SQL de LEFT JOIN conforme pode ser visto nos arquivos X e Y. As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Extração Book NPM" na coluna "Método de extração". 

### - Síntese de variáveis a partir de outras fontes
Para a obtenção de algumas outras variáveis que decidimos utilizar na caracterização do especialista foi necessária a aplicação de algumas etapas de pré processamento antes de podermos usá-las efetivamente na modelagem. Nestes casos, ou o conceito que gostríamos de capturar teve de ser formulado através da agregação de algum dado mais bruto contido em outra tabela, ou o conceito já encontrava-se pronto porém nosso interesse era analisá-lo ao longo do tempo (usando janelas móveis, por exemplo). As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Variável Agregada" na coluna "Método de extração". Para todas estas variáveis é possível consultar AQUI as queries utilizadas em sua criação durante o desenvolvimento, e AQUI as queries que as reproduzem no processo batch de escoragem mensal. 

### - Tratamento de variáveis categóricas
Algumas variáveis categóricas tiveram de ser transformadas em numéricas para darmos cotinuidade ao processo de modelagem. Ainda que os algoritmos que envolvam árvores de decisão e seus derivados (como é o caso do Random Forest, utilizado na modelagem do Angstrom) consigam lidar bem com variáveis categóricas em sua maioria, demos preferência por trata-las manual e individualmente por serem poucas em quantidade e por termos maior controle sobre os possíveis valores finais assumidos pela feature.  


### - Análise de Variance Threshold
Foi realizado um corte das variáveis cuja distribuição de valores apresentavam pouca variância e que, portanto, não acrescentavam informação de grande valor ao modelo em desenvolvimento. Após realizar uma normalização de cada uma das features do conjunto de dados, utilizamos o VarianceThreshold da biblioteca Python scikit-learn e eliminamos as variáveis com variância inferior ao threshold definido de 0.02.


### - Análise visual da distribuição das features
Utilizamos o script do [Pytinela](), que acompanha a estabilidade populacional de uma feature e sua distribuição ao longo das diferentes safras analisadas. Variáveis que apresentam uma alta variabilidade populacional ao longo do tempo tendem a não ser muito indicadas para utilização em modelos preditivos: estes dependem da identificação de padrões nos dados de treino e posterior generalização do aprendizado para o conjunto de dados de teste. Caso a distribuição dos valores de uma feature varie de maneira muito brusca ou sem nenhum padrão discernível, ela pode acabar introduzindo "barulho" nos dados de treino ou até mesmo interagindo com alguma das outras variáveis preditivas, prejudicando a qualidade do modelo final. A partir desta análise acabamos excluíndo mais X variáveis do conjunto final.


### - Análise de correlação entre as variáveis explicativas
Uma outra análise que realizamos com o objetivo de selecionar as variáveis preditivas finais do modelo foi a avaliação do coeficiente de correlação de Pearson para cada par de variáveis candidatas. Os pares de variáveis que apresentaram valor do coeficiente de correlação maior que o threshold de 0.9 foram isolados e, dentre elas, foi selecionada a feature com maior valor de correlação com o target do modelo para permanecer no conjunto final. Nesta etapa, excluímos do conjunto final mais X variáveis, 
restando agora Y variáveis no total.


### - Variáveis finalistas para modelagem
Finalmente, concluídos os passos de pré seleção das variáveis explicativas, obtivemos um conjunto de XYZ variáveis totais, com as quais prosseguimos para a etapa seguinte do processo: a etapa de modelagem. Vale salientar que estas variáveis não necessariamente representam o conjunto de variáveis do modelo final, porém este é um subconjunto daquelas. Mais a frente, conforme detalhado na seção onde discutimos o processo de [Modelagem](), algumas variáveis deste conjunto acabam sendo descartadas após a avaliação do valor de Feature Importance que lhes é atribuído após o ajuste dos dados de treino ao modelo. 


### - Geração da Flat Table de Modelagem
Para as XYZ variáveis pré selecionadas, desenvolvemos um pipeline de extração e cruzamento com o público final para obtermos a flat table que serve de entrada para o processo de modelagem. O passo a passo do processo, que consiste em cruzamentos (JOINs SQL) da tabela de público com as devidas tabelas origem de cada dado, pode ser consultado [neste Jupyter Notebook](). 




# MODELAGEM
---
## Treinamento do modelo
### - Testes iniciais de modelagem
Com a [flat table de modelagem]() em mãos, realizamos alguns experimentos de modelagem utilizando métodos estatísticos e/ou algoritmos de Machine Learning que avaliamos pertinentes ao contexto do problema e ao conjunto de dados que tínhamos em mãos. 

Um dos primeiros testes que realizamos foi a utilização de um emsemble de árvores de decisão na tentativa de modelar os nossos dados. Mais especificamente, testamos a implementação da biblioteca Python [LightGBM](), em razão de já possuirmos bastante familiaridade com sua API e tmabém por já termos grande parte dos pipelines de treinamento e avaliação dos modelos criados a partir desta ferramenta bastante desenvolvidos e maduros em nossa equipe. Os experimentos, porém, não geraram resultados muito positivos: pudemos observar que, mesmo após o ajuste de hiperparâmetros, o modelo tendia a gerar um "overfit" bastante pronunciado dos dados de treino, conforme visto na curva de aprendizagem da Figura XX. A Figura mostra a evolução da função de custo nos dados de teste e validação do modelo após cada iteração do algoritmo (ou, neste caso, após a adição de cada árvore ao emsemble) e nos permite observar que, mesmo após a curva referente aos dados de validação já ter atingido seu mínimo, a curva dos dados de teste continua sua tendência decrescente indefinidamente. Este resultado nos mostra que a quantidade ideal de árvores no emsemble deveria ser YY (valor onde o mínimo da curva dos dados de validação é obtido) mas este valor além de se tratar de um número baixo de árvores também está muito distante do valor da função de custo obtida pelos dados de treinamento, nos levando a concluir, portante, que o algoritmo de Gradient Boosting é complexo demais (ou introduz muita variância) para modelar o nosso conjunto de dados que, como já falamos algumas vezes, possui poucas observações. Partimos então para experimentos com algoritmos mais simples, com menor quantidade de hiperparâmetros e complexidade no geral.

No outro extremo do espectro dos modelos de árvore, fomos testar as Árvores de Decisão simples. Construímos diversas árvores de decisão controlando, a princípio, a quantidade de níveis de profundidade ou a quantidade de folhas finais da árvore. Destes experimentos conseguimos observar que uma profundidade de 4 nívies, ou uma árvore com um máximo de 16 folhas, conseguia produzir valores de Gini que nos indicavam uma boa capacidade de discriminação (alto valor de Gini no conjunto de dados de validação) porém sem o "overfit" do conjunto de dos que observamos com o LGBM, evidenciado pela pequena diferença entre os valores de Gini para o conjunto de treino e de validação. Foram testadas algumas diferentes combinações de árvores e um apanhado dos resultados pode ser observado na Tabela XX.

Procurando um meio termo entre os emsembles gerados pelo método de Gradient Boosting (LGBM) e as Árvores de Decisão clássicas, decidimos testar o algoritmo de Random Forest, obtendo resultados bastante positivos desde o princípio. Alto poder de discriminação aliado a baixos indícios de "overfit" fizeram com a técnica acabasse sendo selecionada para gerar nosso modelo final. 



### - Seleção do modelo final
Tendo decidido pela utilização do algoritmo de Random Forest, partimos para as etapas finais de modelagem que consistem na otimização dos hiperparâmetros e na obtenção do modelo treinado. Utilizamos a ferramenta [Optuna]() para realizar o processo de otimização e os parâmetros obtidos para o treinamento da versão final do modelo estão expostos na Tabela XY a seguir.

TABELA XY

Vale destacar aqui alguns detalhes do processo de otimização. Primeiramente, o espaço amostral que definimos para exploração dos hiperparâmetros, que pode ser consultado na tabela XZ. A tabela contém o nome de cada um dos parâmetros do algoritmo Random Forest que selecionamos bem como o domínio de valores que definimos para cada um deles. O algoritmo de otimização realiza diversas iterações de amostragem destes parâmetros e para cada conjunto de valores amostrados é treinado um modelo e avaliada sua performance, na busca de maximizar (ou minimizar) uma determinada função objetivo.

Em segundo lugar, o algoritmo de amostragem escolhido, que foi o "Tree-structured Parzen Estimator" ou TPE. Este algoritmo é baseado em um processo de amostragem Bayesiano, onde a cada iteração (ou cada novo passo de exploração do espaço amostral de valores para os parâmetros do modelo) o algoritmo se utiliza das informações obtidas para realizar a escolha da amostra seguinte de maneira mais direcionada a atingir o objetivo de otimização. Se compararmos este funcionamento com o de um algoritmo de amostragem totalmente aleatório, onde a cada nova iteração seria selecionado um conjunto de valores ao acaso para utilização no treinamento do modelo, se torna mais evidente a motivação por trás desta escolha e as vantagens do algoritmo TPE.

O terceiro e último detalhe se refere a função objetivo que definimos como meta para otimização. Na grande maioria das vezes a função objetivo é definida a partir de uma métrica de performance do modelo. No nosso caso e no contexto geral de risco de crédito, normalmente estamos em busca dos parâmetros e do modelo capazes de maximizar o valor do Gini ou do KS quando avaliado no conjunto de dados de validação. Por conta da baixa volumetria de dados de treinamento que possuíamos e da forte tendência que os modelos testados apresentavam de "overfittar" nosso conjunto de dados, resolvemos definir uma função objetivo múltipla ou dupla no nosso caso: buscamos os valores de parâmetros que, ao mesmo tempo, fossem capazes de maximizar o valor de Gini porém minimizando a diferença de Gini observada entre o conjunto de dados de treinamento e de validação. Desta forma, nossa intenção é equilibrar da melhor maneira possível a qualidade/performance do modelo treinado com sua capacidade de generalização em dados desconhecidos quando em ambiente produtivo. 

Uma última etapa realizada para obtermos a versão treinada final do modelo foi a retirada do conjunto de treinamento das features que apresentavam valores de Feature Importance zerados, seguido de uma nova rodada de otimização de hiperparâmetros e retreino do modelo. No detalhe abaixo estão resumidos os dados referentes ao modleo Angstrom de acordo com a versão final implantada.  



### - Avaliação do modelo nos dados de desenvolvimento
        - Valor de gini
          
          
        - Feature importances
        - Gráfico de discriminação do score
 
## Definição dos GHs
### - Obtenção dos grupos homogêneos
### - Distribuição do público nos GHs
### - GHs ordenando inadimplencia

## Validação e Resultados
### - Prevendo o especialista 'ddd'
### - Classificando o especialista 'hhh'
### - Matrizes de cruzamento e ginis

## Controles


